# SA-MTL BERT — Google Colab

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Upload your data to Google Drive at: `MyDrive/mtl-bert/data/`
   - `data/sarcasm/sarcasm.csv`
   - `data/cyberbullying/cyberbullying.csv`
   - `data/emotions/emotions.csv`
3. Run all cells in order.

Checkpoints and results are saved to Drive so they survive session disconnects.

In [ ]:
# Install dependencies (transformers not pre-installed on all Colab versions)
!pip install -q transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Change this if your Drive folder is named differently ──
BASE_DIR      = "/content/drive/MyDrive/mtl-bert"
DATA_DIR      = os.path.join(BASE_DIR, "data")
CKPT_BASE_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR   = os.path.join(BASE_DIR, "results", "mtl-bert")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data dir : {DATA_DIR}")
print(f"Results  : {RESULTS_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import random
import json
import csv
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Dataset helpers (inlined from dataset.py) ──────────────────────────

def create_sample_datasets():
    datasets = {}

    # Sarcasm — CSV: id,class,text
    sarc_path = os.path.join(DATA_DIR, "sarcasm", "sarcasm.csv")
    sarc_data = []
    with open(sarc_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        sarc_data.append((text, label))
                except ValueError:
                    continue
    datasets["sarc"] = sarc_data

    # Cyberbullying — CSV: id,class,text
    cyber_path = os.path.join(DATA_DIR, "cyberbullying", "cyberbullying.csv")
    intent_data = []
    with open(cyber_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        intent_data.append((text, label))
                except ValueError:
                    continue
    datasets["intent"] = intent_data

    # Emotions — CSV: class,text
    emo_path = os.path.join(DATA_DIR, "emotions", "emotions.csv")
    emotion_data = []
    with open(emo_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 2:
                try:
                    label = int(row[0])
                    text  = row[1].strip()
                    if text:
                        emotion_data.append((text, label))
                except ValueError:
                    continue
    datasets["emotion"] = emotion_data

    return datasets


def compute_metrics(predictions, labels):
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, predictions, average="weighted", zero_division=0),
    }


def compute_per_class_metrics(predictions, labels, class_names=None):
    report_str  = classification_report(labels, predictions, target_names=class_names, zero_division=0)
    report_dict = classification_report(labels, predictions, target_names=class_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(labels, predictions)
    return {"report_str": report_str, "report_dict": report_dict, "confusion_matrix": cm.tolist()}

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── SingleTaskDataset (Sec 3.2.5) ─────────────────────────────────────

class SingleTaskDataset(Dataset):
    def __init__(self, data: List[Tuple], tokenizer, max_length=128):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(label, dtype=torch.long)
        }


# ── MultitaskModel with MLP heads (Sec 3.2.3–3.2.4) ──────────────────
# Fix 1: MLP task heads to mitigate negative transfer.
# Each head has a hidden layer (hidden_size/2) with ReLU + dropout,
# giving each task its own learned intermediate representation
# on top of the shared encoder.

class MultitaskModel(nn.Module):
    def __init__(self, model_name: str, task_configs: Dict[str, int], dropout: float = 0.3):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.task_heads = nn.ModuleDict()
        for task_name, num_classes in task_configs.items():
            out = 1 if num_classes == 2 else num_classes
            self.task_heads[task_name] = nn.Sequential(
                nn.Linear(hidden_size, hidden_size // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size // 2, out),
            )
        self.task_configs = task_configs

    def forward(self, input_ids, attention_mask, task_name):
        outputs      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled       = outputs.last_hidden_state[:, 0]  # [CLS]
        return self.task_heads[task_name](pooled)


# ── MultitaskTrainer (Sec 3.2.5, 3.3.3–3.3.4) ────────────────────────

class MultitaskTrainer:
    def __init__(self, model, tokenizer, task_configs, device='cpu', task_sizes=None):
        self.model        = model.to(device)
        self.tokenizer    = tokenizer
        self.task_configs = task_configs
        self.device       = device
        self.ckpt_dir     = None  # set by load_checkpoint
        self.optimizer    = optim.AdamW(self.model.parameters(), lr=2e-5, weight_decay=0.01)

        self.loss_fns = {}
        for task, num_classes in task_configs.items():
            self.loss_fns[task] = nn.BCEWithLogitsLoss() if num_classes == 2 else nn.CrossEntropyLoss()

        # Inverse-sqrt frequency task weighting (Sec 3.3.3)
        if task_sizes is not None:
            raw    = {t: 1.0 / (task_sizes[t] ** 0.5) for t in task_configs}
            mean_w = sum(raw.values()) / len(raw)
            self.task_weights = {t: w / mean_w for t, w in raw.items()}
            print("  Task weights (inverse-sqrt frequency, Sec 3.3.3):")
            for t, w in self.task_weights.items():
                print(f"    {t}: {w:.4f}  (N_train={task_sizes[t]:,})")
        else:
            self.task_weights = {t: 1.0 for t in task_configs}

        self.history = {
            'train_loss' : [],
            'task_losses': {t: [] for t in task_configs},
            'val_metrics': {t: [] for t in task_configs},
        }
        self.start_epoch = 0

        # Best model selection: track the best average validation F1 across tasks
        self.best_val_f1 = 0.0
        self.best_model_state = None

    def save_checkpoint(self, epoch, checkpoint_dir, mid_epoch=False):
        os.makedirs(checkpoint_dir, exist_ok=True)
        ckpt = {
            'epoch'               : epoch,
            'mid_epoch'           : mid_epoch,
            'model_state_dict'    : self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'history'             : self.history,
            'task_weights'        : self.task_weights,
            'best_val_f1'         : self.best_val_f1,
            'best_model_state'    : self.best_model_state,
        }
        if not mid_epoch:
            path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pt")
            torch.save(ckpt, path)
            print(f"  Checkpoint saved: {path}")
        latest = os.path.join(checkpoint_dir, "checkpoint_latest.pt")
        torch.save(ckpt, latest)
        if mid_epoch:
            print(f"  Mid-epoch checkpoint saved")

    def load_checkpoint(self, checkpoint_dir):
        self.ckpt_dir = checkpoint_dir  # store so train_epoch can use it
        latest = os.path.join(checkpoint_dir, "checkpoint_latest.pt")
        if not os.path.exists(latest):
            return False
        print(f"\nFound checkpoint at {latest}, resuming training...")
        try:
            ckpt = torch.load(latest, map_location=self.device, weights_only=False)
            self.model.load_state_dict(ckpt['model_state_dict'])
            self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            self.history = ckpt['history']
            self.best_val_f1 = ckpt.get('best_val_f1', 0.0)
            self.best_model_state = ckpt.get('best_model_state', None)
            if ckpt.get('mid_epoch', False):
                self.start_epoch = ckpt['epoch']
                print(f"  Resumed mid-epoch {ckpt['epoch']+1}. Will restart epoch {self.start_epoch+1}.")
            else:
                self.start_epoch = ckpt['epoch'] + 1
                print(f"  Resumed after epoch {ckpt['epoch']+1}. Will continue from epoch {self.start_epoch+1}.")
            if self.best_val_f1 > 0:
                print(f"  Best Val F1 so far: {self.best_val_f1:.4f}")
        except Exception as e:
            print(f"  WARNING: Checkpoint corrupted ({e}). Deleting and starting from scratch.")
            os.remove(latest)
            return False
        return True

    def update_best_model(self, val_metrics):
        """Track best model by average validation F1 across all tasks."""
        avg_f1 = np.mean([m['f1'] for m in val_metrics.values()])
        if avg_f1 > self.best_val_f1:
            self.best_val_f1 = avg_f1
            self.best_model_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            print(f"  New best model: avg Val F1 = {avg_f1:.4f}")
        return avg_f1

    def load_best_model(self):
        """Load the best model state for test evaluation."""
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            self.model.to(self.device)
            print(f"  Loaded best model (avg Val F1 = {self.best_val_f1:.4f})")
        else:
            print("  No best model saved, using final epoch model")

    # Fix 2: Dataloader cycling — smaller dataloaders restart instead of
    # being exhausted, so all tasks receive gradient updates throughout
    # the entire epoch.
    def train_epoch(self, task_dataloaders, epoch):
        self.model.train()
        total_loss   = 0
        task_losses  = {t: 0 for t in self.task_configs}
        task_counts  = {t: 0 for t in self.task_configs}
        task_iters   = {t: iter(dl) for t, dl in task_dataloaders.items()}
        task_names   = list(task_dataloaders.keys())
        # Total steps = largest dataloader * num_tasks (equal turns per task)
        max_batches  = max(len(dl) for dl in task_dataloaders.values())
        total_steps  = max_batches * len(task_names)
        step, task_idx = 0, 0

        while step < total_steps:
            task_name  = task_names[task_idx % len(task_names)]
            task_idx  += 1
            try:
                batch = next(task_iters[task_name])
            except StopIteration:
                # Cycle: restart exhausted dataloaders so smaller tasks
                # continue receiving gradient updates throughout the epoch
                task_iters[task_name] = iter(task_dataloaders[task_name])
                batch = next(task_iters[task_name])

            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['label'].to(self.device)
            logits         = self.model(input_ids, attention_mask, task_name)

            if self.task_configs[task_name] == 2:
                loss = self.loss_fns[task_name](logits.squeeze(-1), labels.float())
            else:
                loss = self.loss_fns[task_name](logits, labels)

            weighted_loss = loss * self.task_weights[task_name]
            self.optimizer.zero_grad()
            weighted_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss             += weighted_loss.item()
            task_losses[task_name] += loss.item()
            task_counts[task_name] += 1
            step                   += 1

            if step % 10 == 0:
                print(f'Epoch {epoch}, Step {step}/{total_steps}, Task: {task_name}, Loss: {loss.item():.4f}')
            if step % 1000 == 0 and self.ckpt_dir:
                self.save_checkpoint(epoch, checkpoint_dir=self.ckpt_dir, mid_epoch=True)

        avg_total = total_loss / max(step, 1)
        avg_tasks = {t: task_losses[t] / max(task_counts[t], 1) for t in self.task_configs}
        self.history['train_loss'].append(avg_total)
        for t, l in avg_tasks.items():
            self.history['task_losses'][t].append(l)
        return avg_total, avg_tasks

    def evaluate(self, task_dataloaders):
        self.model.eval()
        preds  = {t: [] for t in self.task_configs}
        lbls   = {t: [] for t in self.task_configs}
        with torch.no_grad():
            for task_name, dl in task_dataloaders.items():
                for batch in dl:
                    input_ids      = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    logits         = self.model(input_ids, attention_mask, task_name)
                    if self.task_configs[task_name] == 2:
                        predictions = (torch.sigmoid(logits.squeeze(-1)) > 0.5).long()
                    else:
                        predictions = torch.argmax(logits, dim=-1)
                    preds[task_name].extend(predictions.cpu().numpy())
                    lbls[task_name].extend(batch['label'].numpy())

        task_metrics = {}
        for t in self.task_configs:
            if preds[t]:
                m = compute_metrics(preds[t], lbls[t])
                task_metrics[t] = m
                self.history['val_metrics'][t].append(m)
        return task_metrics


def load_json(path):
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)

In [ ]:
# ── Main training ──────────────────────────────────────────────────────

print("=" * 60)
print("  SA-MTL Framework (Sec 3.5.3) - Multi-Seed Training")
print("=" * 60)

MODEL_NAME     = "bert-base-uncased"
SEEDS          = [42, 123, 456]
BATCH_SIZE     = 16          # T4 has 16 GB — safe to use 16 vs 8 on your local GTX 1650
NUM_EPOCHS     = 5
MAX_LENGTH     = 128
EMOTION_CLASSES = ["sad", "joy", "love", "angry", "fear", "surprise"]

task_configs = {'sarc': 2, 'intent': 2, 'emotion': 6}

print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nLoading datasets...")
tasks_data = create_sample_datasets()
for task_name, data in tasks_data.items():
    print(f"  {task_name}: {len(data)} samples")

# Resume support
progress_path       = os.path.join(RESULTS_DIR, "training_progress.json")
all_seed_results    = {t: [] for t in task_configs}
all_seed_predictions = {t: [] for t in task_configs}
all_seed_labels     = {t: [] for t in task_configs}
all_seed_histories  = []
completed           = set()

if os.path.exists(progress_path):
    progress = load_json(progress_path)
    if progress:
        completed            = set(progress.get("completed", []))
        all_seed_results     = progress.get("results",     all_seed_results)
        all_seed_predictions = progress.get("predictions", all_seed_predictions)
        all_seed_labels      = progress.get("labels",      all_seed_labels)
        all_seed_histories   = progress.get("histories",   [])
        if completed:
            print(f"\nResuming: {len(completed)} seed(s) already done.")

for seed_idx, seed in enumerate(SEEDS):
    seed_key = f"seed{seed}"
    if seed_key in completed:
        print(f"\n--- Skipping seed {seed} (already done) ---")
        continue

    print(f"\n{'='*60}")
    print(f"  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})")
    print(f"{'='*60}")
    set_seed(seed)

    train_data, val_data, test_data = {}, {}, {}
    for task_name, samples in tasks_data.items():
        shuffled = samples.copy()
        random.shuffle(shuffled)
        n  = len(shuffled)
        s1, s2 = int(0.8 * n), int(0.9 * n)
        train_data[task_name] = shuffled[:s1]
        val_data[task_name]   = shuffled[s1:s2]
        test_data[task_name]  = shuffled[s2:]

    train_loaders, val_loaders, test_loaders = {}, {}, {}
    for task_name in tasks_data:
        train_loaders[task_name] = DataLoader(SingleTaskDataset(train_data[task_name], tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
        val_loaders[task_name]   = DataLoader(SingleTaskDataset(val_data[task_name],   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
        test_loaders[task_name]  = DataLoader(SingleTaskDataset(test_data[task_name],  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

    model = MultitaskModel(MODEL_NAME, task_configs)
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

    task_train_sizes = {t: len(train_data[t]) for t in tasks_data}
    trainer  = MultitaskTrainer(model, tokenizer, task_configs, device, task_sizes=task_train_sizes)
    ckpt_dir = os.path.join(CKPT_BASE_DIR, f"mtl-bert/{seed_key}")
    trainer.load_checkpoint(ckpt_dir)

    print(f"\n  Training for {NUM_EPOCHS} epochs...")
    for epoch in range(trainer.start_epoch, NUM_EPOCHS):
        print(f"\n  Epoch {epoch+1}/{NUM_EPOCHS}")
        train_loss, task_losses_ep = trainer.train_epoch(train_loaders, epoch)
        print(f"  Train Loss: {train_loss:.4f}  "
              f"(sarc={task_losses_ep['sarc']:.4f}, "
              f"intent={task_losses_ep['intent']:.4f}, "
              f"emotion={task_losses_ep['emotion']:.4f})")
        val_metrics = trainer.evaluate(val_loaders)
        for t, m in val_metrics.items():
            print(f"    {t}: Acc={m['accuracy']:.4f}, F1={m['f1']:.4f}")

        # Track best model by average validation F1
        trainer.update_best_model(val_metrics)
        trainer.save_checkpoint(epoch, ckpt_dir)

    # Load best model for test evaluation
    trainer.load_best_model()

    print(f"\n  Test Evaluation (seed={seed}):")
    test_metrics = trainer.evaluate(test_loaders)

    model.eval()
    seed_preds = {t: [] for t in task_configs}
    seed_lbls  = {t: [] for t in task_configs}
    with torch.no_grad():
        for task_name, dl in test_loaders.items():
            for batch in dl:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                logits         = model(input_ids, attention_mask, task_name)
                if task_configs[task_name] == 2:
                    p = (torch.sigmoid(logits.squeeze(-1)) > 0.5).long()
                else:
                    p = torch.argmax(logits, dim=-1)
                seed_preds[task_name].extend(p.cpu().numpy().tolist())
                seed_lbls[task_name].extend(batch['label'].numpy().tolist())

    for t, m in test_metrics.items():
        all_seed_results[t].append(m)
        all_seed_predictions[t].append(seed_preds[t])
        all_seed_labels[t].append(seed_lbls[t])
        print(f"    {t}: Acc={m['accuracy']:.4f}, P={m['precision']:.4f}, "
              f"R={m['recall']:.4f}, F1={m['f1']:.4f}")

    all_seed_histories.append(trainer.history)
    completed.add(seed_key)
    with open(progress_path, "w") as f:
        json.dump({"completed": list(completed), "results": all_seed_results,
                   "predictions": all_seed_predictions, "labels": all_seed_labels,
                   "histories": all_seed_histories}, f, indent=2)
    print(f"  Progress saved ({len(completed)}/{len(SEEDS)} seeds done)")
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"mtl_model_{seed_key}.pt"))

In [ ]:
# ── Aggregated results ─────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"  Aggregated Results (mean +/- std, {len(SEEDS)} seeds)")
print(f"{'='*60}")

aggregated = {}
for task in task_configs:
    agg = {}
    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = [m[metric] for m in all_seed_results[task]]
        agg[metric] = {"mean": float(np.mean(values)), "std": float(np.std(values)),
                       "per_seed": [float(v) for v in values]}
    aggregated[task] = agg
    print(f"\n  {task}:")
    for metric in ["accuracy", "precision", "recall", "f1"]:
        print(f"    {metric:>10s}: {agg[metric]['mean']:.4f} +/- {agg[metric]['std']:.4f}")

with open(os.path.join(RESULTS_DIR, "aggregated_results.json"), "w") as f:
    json.dump(aggregated, f, indent=2)

# Per-class emotion metrics
best_idx   = int(np.argmax([m["f1"] for m in all_seed_results["emotion"]]))
per_class  = compute_per_class_metrics(
    all_seed_predictions["emotion"][best_idx],
    all_seed_labels["emotion"][best_idx],
    EMOTION_CLASSES
)
print(f"\n  Per-Class Emotion (best seed: {SEEDS[best_idx]}):")
print(per_class["report_str"])
per_class_save = dict(per_class["report_dict"])
per_class_save["confusion_matrix"] = per_class["confusion_matrix"]
with open(os.path.join(RESULTS_DIR, "emotion_per_class_metrics.json"), "w") as f:
    json.dump(per_class_save, f, indent=2)

# Training plots
if all_seed_histories:
    best_history = all_seed_histories[best_idx]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    n_epochs = len(best_history['train_loss'])
    axes[0].plot(range(1, n_epochs+1), best_history['train_loss'], marker='o', label='Total Loss')
    for t in task_configs:
        axes[0].plot(range(1, n_epochs+1), best_history['task_losses'][t], marker='s', label=f'{t} Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('MTL Training Loss per Epoch'); axes[0].legend(); axes[0].grid(True)
    for t in task_configs:
        f1s = [m['f1'] for m in best_history['val_metrics'][t]]
        axes[1].plot(range(1, len(f1s)+1), f1s, marker='o', label=f'{t} F1')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
    axes[1].set_title('MTL Validation F1 per Epoch'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "training_plots.png"), dpi=150)
    plt.show()
    print(f"Plots saved to {RESULTS_DIR}/training_plots.png")

print("\n  Done!")